# exp-back — worked example 3: Compose exp_back with a Multiply Backward

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `exp-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a real backward pass, each backward function receives the upstream gradient (output of the next backward function in the chain) as its `grad_out`. When composing two ops such as `y = exp(3 * x)`, the chain rule is: `dL/dx = dL/dy * d(exp(u))/du * d(3x)/dx = grad_out * exp(3x) * 3`. This can be computed step by step using `exp_back` and a multiply backward.

## Worked solution

We compute the gradient of `loss = exp(3 * x).sum()` through a manual two-step backward pass.

**Forward:** `u = 3 * x` (multiply by constant), then `y = exp(u)`.

**Reverse step 1 — exp_back:** `g_u = exp_back(ones_like(y), y, u) = ones * y = exp(3x)`.

**Reverse step 2 — multiply_back:** The backward of `u = c * x` is `g_x = g_u * c = exp(3x) * 3`.

**Verify:** This should match `d/dx exp(3x) = 3 * exp(3x)`. We confirm against `torch.autograd`.

In [ ]:
import torch as t
from torch import Tensor

t.manual_seed(21)

def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return grad_out * out

def mul_const_back(grad_out: Tensor, c: float) -> Tensor:
    """Backward of y = c * x: dL/dx = grad_out * c."""
    return grad_out * c

# Forward
c = 3.0
x = t.tensor([0.5, 1.0, -0.5, 2.0])
u = c * x                   # u = 3x
y = t.exp(u)                # y = exp(3x)

# Backward
g_u = exp_back(t.ones_like(y), y, u)       # = exp(3x)
g_x = mul_const_back(g_u, c)               # = exp(3x) * 3

print(f"x:   {x.tolist()}")
print(f"g_x: {g_x.tolist()}")
print(f"expected (3*exp(3x)): {(c * y).tolist()}")
assert t.allclose(g_x, c * y), "Computed gradient doesn't match 3*exp(3x)"

# Verify against autograd
xa = x.clone().requires_grad_(True)
t.exp(c * xa).sum().backward()
assert t.allclose(g_x, xa.grad)
print("Composed exp_back + mul_const_back matches torch.autograd.")